# Data Prcessing Pipeline

## Goal
Convert raw audio files into standardized, normalized feature tensors ready for CNN training.

## Pipeline Steps
1.  **Load Audio**: Load with original sampling rate.
2.  **Resample**: Convert to `22050 Hz`.
3.  **Fix Duration**: Pad or Trim to exactly `2 seconds` (44100 samples).
4.  **Feature Extraction**: Generate Mel-Spectrogram (`n_mels=128`).
5.  **Log Scaling**: Convert power to dB.
6.  **Normalization**: Standardize features.
7.  **Tensor Format**: Reshape to `(1, 128, T)`.
8.  **Output**: Save `X.npy` and `y.npy`.

In [20]:
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

# Settings
TARGET_SR = 22050
DURATION = 2 # Seconds
SAMPLES = int(TARGET_SR * DURATION) # 44100
Result_Path = '../data/processed/'
META_PATH = '../data/processed/master_metadata.csv'
N_MELS = 128

# Ensure output dir exists
os.makedirs(Result_Path, exist_ok=True)

In [21]:
# Load Metadata
df = pd.read_csv(META_PATH)
print(f"Total files to process: {len(df)}")

# Encode Labels
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['class'])

# Save classes mapping
np.save(os.path.join(Result_Path, 'classes.npy'), le.classes_)
print(f"Classes: {le.classes_}")

Total files to process: 920
Classes: ['Car Horn' 'Crying Baby' 'Dog Bark' 'Door Knock' 'Siren']


## Processing Function

In [22]:
def process_audio(file_path):
    try:
        # 1. Load Audio (Original SR)
        y, sr = librosa.load(file_path, sr=None)
        
        # 2. Resample if needed
        if sr != TARGET_SR:
            y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
            
        # 3. Fix Duration (Pad/Trim)
        if len(y) > SAMPLES:
            y = y[:SAMPLES]
        else:
            padding = SAMPLES - len(y)
            y = np.pad(y, (0, padding), mode='constant')
            
        # 4. Mel Spectrogram
        mel = librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_mels=N_MELS)
        
        # 5. Log Scaling
        log_mel = librosa.power_to_db(mel, ref=np.max)
        
        # 6. Normalization (Per-sample Z-score)
        # Check for essentially silent audio to avoid divide by zero
        if np.std(log_mel) > 1e-6:
            log_mel = (log_mel - np.mean(log_mel)) / np.std(log_mel)
        else:
            log_mel = np.zeros_like(log_mel)
            
        # 7. Convert to Tensor Format (1, Mel, Time)
        # Current shape is (Mel, Time). Add channel dim.
        # For PyTorch/Keras, usually (Channels, Height, Width) or (Height, Width, Channels)
        # User requested: (1, 128, T)
        tensor = np.expand_dims(log_mel, axis=0)
        
        return tensor
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

## Execution Loop

In [23]:
features = []
labels = []

print("Starting Feature Extraction...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    path = row['filepath']
    label = row['label_encoded']
    
    feat = process_audio(path)
    
    if feat is not None:
        features.append(feat)
        labels.append(label)#

print(len(features))
print(features[0].shape)

print(features[0])
print(labels[0])


X = np.array(features)
y = np.array(labels)

print(f"Final X shape: {X.shape}")
print(f"Final y shape: {y.shape}")

Starting Feature Extraction...


100%|██████████| 920/920 [00:03<00:00, 271.91it/s]

920
(1, 128, 87)
[[[ 0.93418545  0.6575904   0.5906572  ...  1.0491847   0.9756285
    1.316856  ]
  [ 0.91432786  0.68024635  0.45129028 ...  0.7721654   1.1664242
    1.4636676 ]
  [ 0.8497145   0.62522924  0.17622444 ...  1.0543103   1.2992096
    1.3870881 ]
  ...
  [-0.17593513 -0.28168532 -0.5538083  ...  0.05814619  0.39257106
    0.65802574]
  [-0.1928421  -0.29542342 -0.5605218  ... -0.15535934  0.2985233
    0.59783465]
  [-0.57636434 -0.6605026  -0.84243304 ... -0.73111326  0.14461546
    0.5824902 ]]]
2
Final X shape: (920, 1, 128, 87)
Final y shape: (920,)


## Save Results

In [24]:
np.save(os.path.join(Result_Path, 'X.npy'), X)
np.save(os.path.join(Result_Path, 'y.npy'), y)

print("Feature extraction complete. Files saved to data/processed/")

Feature extraction complete. Files saved to data/processed/


In [25]:
print(np.min(X), np.max(X))
print(np.mean(X), np.std(X))
print(np.isnan(X).any())

-5.1799874 7.793202
-4.9208833e-09 1.0
False


mean ≈ 0
Нормализацията работи.

std = 1
Z-score е стабилен.

няма NaN

Няма silent-corruption.

min/max не са проблем

Z-score няма фиксиран диапазон.
+7 или -5  напълно нормално при аудио.